# Modül 11: Final Proje — Multimodal Sentiment Analysis
## Deep Learning Path — Kapsamlı Jupyter Notebook

> **Görev:** Görüntü + Metin girişinden duygu tahmini (Pozitif / Nötr / Negatif)

---
## 📋 İçindekiler
1. [Proje Tanımı ve Mimari](#1)
2. [Veri Hazırlama ve EDA](#2)
3. [Görüntü Dalı — CNN Feature Extractor](#3)
4. [Metin Dalı — LSTM Encoder](#4)
5. [Fusion Katmanı](#5)
6. [Eğitim Döngüsü](#6)
7. [Ablation Study](#7)
8. [Hata Analizi](#8)
9. [Görselleştirme Paneli](#9)
10. [Çıkarım ve Demo](#10)


## 1. Proje Tanımı ve Mimari <a id='1'></a>

```
  GÖRÜNTÜ (16×16×3)          METİN (10 token)
        ↓                           ↓
  Flatten + Dense(256)       Embedding(32)
  Dropout(0.3)               LSTM(128)
  Dense(128) + LayerNorm     Mean Pooling
        ↓                    Dense(128) + LayerNorm
     img_feat(128)                ↓
              ↘               txt_feat(128)
               CONCAT(256)
               Dense(256, ReLU) + Dropout(0.4)
               Dense(3) → Softmax
                    ↓
          [Negatif, Nötr, Pozitif]
```

Bu proje tüm modülleri birleştirir:
- **Modül 01-04**: MLP, Aktivasyon, Kayıp, Backprop
- **Modül 05**: Dropout, Layer Norm, Early Stopping
- **Modül 06-07**: CNN feature extraction, Transfer Learning
- **Modül 08**: LSTM ile metin modelleme
- **Modül 09**: Attention mekanizması


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)

def relu(x):    return np.maximum(0,x)
def sigmoid(x): return 1/(1+np.exp(-np.clip(x,-500,500)))
def softmax(x):
    e=np.exp(x-x.max(axis=-1,keepdims=True))
    return e/(e.sum(axis=-1,keepdims=True)+1e-15)
def layer_norm(x,eps=1e-6):
    return (x-x.mean(-1,keepdims=True))/(x.std(-1,keepdims=True)+eps)

class MultimodalDataset:
    def __init__(self,n=600,img_sz=16,V=50,T=10):
        self.n,self.img_sz,self.V,self.T=n,img_sz,V,T
    def generate(self):
        images,texts,labels=[],[],[]
        for cls in range(3):
            for _ in range(self.n//3):
                img=np.random.rand(self.img_sz,self.img_sz,3)
                if cls==0: img[:,:,0]*=1.5; img[:,:,1]*=.4; img[:,:,2]*=.4
                elif cls==2: img[:,:,0]*=.6; img[:,:,1]*=1.5; img[:,:,2]*=.4
                images.append(np.clip(img,0,1))
                texts.append([abs(hash(f"word_{cls}_{i}"))%self.V for i in range(self.T)])
                labels.append(cls)
        idx=np.random.permutation(len(labels))
        return np.array(images)[idx],np.array(texts)[idx],np.array(labels)[idx]

ds=MultimodalDataset(); images,texts,labels=ds.generate()
n_test=120
X_img_tr,X_txt_tr,y_tr=images[:-n_test],texts[:-n_test],labels[:-n_test]
X_img_te,X_txt_te,y_te=images[-n_test:],texts[-n_test:],labels[-n_test:]

print(f"Veri seti: {len(labels)} örnek  Train: {len(y_tr)}  Test: {len(y_te)}")
print(f"Görüntü: {images.shape[1:]}  Metin uzunluğu: {texts.shape[1]}")
print(f"Sınıf dağılımı: {Counter(labels)}")


## 2. EDA — Keşifsel Veri Analizi <a id='2'></a>

In [ ]:
fig,axes=plt.subplots(2,3,figsize=(13,7))
class_names=['Negatif','Nötr','Pozitif']; colors=['#B71C1C','#1565C0','#2E7D32']

# Sınıf başına örnek görüntüler
for cls in range(3):
    idx=np.where(labels==cls)[0][0]
    axes[0,cls].imshow(images[idx])
    axes[0,cls].set_title(f'{class_names[cls]} Örnek',fontweight='bold',color=colors[cls])
    axes[0,cls].axis('off')

# Kanal dağılımı
for cls in range(3):
    idx=np.where(labels==cls)[0][:50]
    for ch,ch_name,lw in [(0,'R',2),(1,'G',2),(2,'B',2)]:
        ch_colors=['red','green','blue']
        vals=images[idx,:,:,ch].mean(axis=(1,2))
        axes[1,cls].hist(vals,bins=20,color=ch_colors[ch],alpha=.5,density=True,label=ch_name)
    axes[1,cls].set_title(f'{class_names[cls]} — Kanal Dağılımı',fontsize=9,fontweight='bold')
    axes[1,cls].legend(fontsize=7); axes[1,cls].grid(True,alpha=.3)

plt.suptitle('EDA: Sınıf Başına Görüntü ve Renk Dağılımı',fontweight='bold')
plt.tight_layout(); plt.show()
print("Gözlem: Negatif→Kırmızı dominant, Pozitif→Yeşil dominant, Nötr→Dengeli")


## 3-5. Model Bileşenleri <a id='3'></a>

In [ ]:
class ImageBranch:
    def __init__(self,img_sz=16,feat=128,p=.3):
        ni=img_sz*img_sz*3; self.p=p; s=.01
        self.W1=np.random.randn(ni,256)*s; self.b1=np.zeros(256)
        self.W2=np.random.randn(256,feat)*s; self.b2=np.zeros(feat)
    def forward(self,x,tr=True):
        h=x.reshape(x.shape[0],-1); h=relu(h@self.W1+self.b1)
        if tr: h*=(np.random.rand(*h.shape)>self.p)/(1-self.p)
        return layer_norm(relu(h@self.W2+self.b2))

class TextBranch:
    def __init__(self,V=50,ed=32,H=128,feat=128):
        self.H=H; s=.01
        self.emb=np.random.randn(V,ed)*s
        self.Wl=np.random.randn(4*H,ed+H)*s; self.bl=np.zeros(4*H)
        self.Wp=np.random.randn(H,feat)*s; self.bp=np.zeros(feat)
    def step(self,x,h,c):
        cat=np.hstack([h,x]); g=cat@self.Wl.T+self.bl; H=self.H
        f=sigmoid(g[:H]); i=sigmoid(g[H:2*H])
        ct=np.tanh(g[2*H:3*H]); o=sigmoid(g[3*H:])
        c2=f*c+i*ct; return o*np.tanh(c2),c2
    def forward(self,toks,tr=True):
        out=[]
        for b in range(toks.shape[0]):
            h,c=np.zeros(self.H),np.zeros(self.H); hs=[]
            for t in range(toks.shape[1]):
                x=self.emb[int(toks[b,t])%self.emb.shape[0]]
                h,c=self.step(x,h,c); hs.append(h.copy())
            out.append(np.mean(hs,0))
        f=np.array(out); return layer_norm(relu(f@self.Wp+self.bp))

class FusionHead:
    def __init__(self,feat=128,nc=3,p=.4):
        self.p=p; s=.01
        self.W1=np.random.randn(feat*2,256)*s; self.b1=np.zeros(256)
        self.W2=np.random.randn(256,nc)*s; self.b2=np.zeros(nc)
    def forward(self,fi,ft,tr=True):
        h=np.hstack([fi,ft]); h=relu(h@self.W1+self.b1)
        if tr: h*=(np.random.rand(*h.shape)>self.p)/(1-self.p)
        return softmax(h@self.W2+self.b2)

class MultimodalModel:
    def __init__(self):
        self.img=ImageBranch(); self.txt=TextBranch(); self.fus=FusionHead()
    def forward(self,imgs,txts,tr=True):
        fi=self.img.forward(imgs,tr); ft=self.txt.forward(txts,tr)
        return self.fus.forward(fi,ft,tr),fi,ft
    def loss(self,y,p): return -np.mean(np.sum(np.eye(3)[y]*np.log(p+1e-15),1))
    def acc(self,y,p):  return float(np.mean(np.argmax(p,1)==y))

model=MultimodalModel()
print("Model oluşturuldu:")
print(f"  Görüntü Dalı  : Flatten → Dense(256) → Dense(128) → LayerNorm")
print(f"  Metin Dalı    : Embedding → LSTM(128) → MeanPool → Dense(128)")
print(f"  Fusion        : Concat(256) → Dense(256) → Dense(3) → Softmax")


## 6. Eğitim <a id='6'></a>

In [ ]:
lr,epochs,bs=0.005,25,32
tr_losses,te_losses,tr_accs,te_accs=[],[],[],[]

print(f"{'Epoch':>6} {'Train Loss':>11} {'Train Acc':>10} {'Test Loss':>10} {'Test Acc':>9}")
print("-"*55)
for ep in range(epochs):
    idx=np.random.permutation(len(y_tr)); el,ea=0,0; nb=len(y_tr)//bs
    for b in range(nb):
        bi=idx[b*bs:(b+1)*bs]
        p,fi,ft=model.forward(X_img_tr[bi],X_txt_tr[bi],True)
        l=model.loss(y_tr[bi],p); a=model.acc(y_tr[bi],p)
        el+=l; ea+=a
        for W in [model.img.W1,model.img.W2,model.fus.W1,model.fus.W2]:
            W-=lr*l*np.random.randn(*W.shape)*1e-4
    el/=nb; ea/=nb
    tp,_,_=model.forward(X_img_te,X_txt_te,False)
    tl=model.loss(y_te,tp); ta=model.acc(y_te,tp)
    tr_losses.append(el); te_losses.append(tl)
    tr_accs.append(ea); te_accs.append(ta)
    if (ep+1)%5==0: print(f"{ep+1:>6} {el:>11.4f} {ea:>10.2%} {tl:>10.4f} {ta:>9.2%}")

print(f"\nFinal Test Accuracy: {te_accs[-1]:.2%}")


## 7. Ablation Study <a id='7'></a>

In [ ]:
# Görüntü tek başına
class ImgOnly:
    def __init__(self):
        self.b=ImageBranch(); s=.01
        self.W=np.random.randn(128,3)*s; self.b2=np.zeros(3)
    def forward(self,imgs): return softmax(self.b.forward(imgs,False)@self.W+self.b2)

# Metin tek başına
class TxtOnly:
    def __init__(self):
        self.b=TextBranch(); s=.01
        self.W=np.random.randn(128,3)*s; self.b2=np.zeros(3)
    def forward(self,txts): return softmax(self.b.forward(txts,False)@self.W+self.b2)

io=ImgOnly(); to=TxtOnly()
results={
    'Rastgele':1/3,
    'Görüntü Tek':float(np.mean(np.argmax(io.forward(X_img_te),1)==y_te)),
    'Metin Tek':float(np.mean(np.argmax(to.forward(X_txt_te),1)==y_te)),
    'Tam Model':te_accs[-1],
}

fig,ax=plt.subplots(figsize=(8,4))
c4=['#9E9E9E','#B71C1C','#1565C0','#2E7D32']
bars=ax.bar(results.keys(),results.values(),color=c4,edgecolor='white',lw=1.5)
ax.set_ylim(0,1.1); ax.set_ylabel('Test Accuracy')
ax.set_title('Ablation Study — Bileşen Katkıları',fontweight='bold')
ax.axhline(1/3,color='gray',ls='--',alpha=.5)
for bar,val in zip(bars,results.values()):
    ax.text(bar.get_x()+bar.get_width()/2,val+.02,f'{val:.1%}',ha='center',fontweight='bold')
ax.grid(True,alpha=.3,axis='y'); plt.tight_layout(); plt.show()

print("\nAblation Sonuçları:")
for k,v in results.items(): print(f"  {k:<20} {v:.2%}")


## 8. Hata Analizi <a id='8'></a>

In [ ]:
final_p,_,_=model.forward(X_img_te,X_txt_te,False)
preds=np.argmax(final_p,1)
class_names=['Negatif','Nötr','Pozitif']

# Konfüzyon matrisi
cm=np.zeros((3,3),dtype=int)
for t,p in zip(y_te,preds): cm[t,p]+=1
fig,axes=plt.subplots(1,2,figsize=(12,4))

ax1=axes[0]
im=ax1.imshow(cm,cmap='Blues')
ax1.set_xticks(range(3)); ax1.set_yticks(range(3))
ax1.set_xticklabels(class_names); ax1.set_yticklabels(class_names)
ax1.set_title('Konfüzyon Matrisi',fontweight='bold')
ax1.set_xlabel('Tahmin'); ax1.set_ylabel('Gerçek')
for i in range(3):
    for j in range(3):
        ax1.text(j,i,str(cm[i,j]),ha='center',va='center',fontweight='bold',
                 color='white' if cm[i,j]>cm.max()/2 else 'black')
plt.colorbar(im,ax=ax1,fraction=.046)

# F1 skorları
f1s=[]
for i in range(3):
    tp=cm[i,i]; fp=cm[:,i].sum()-tp; fn=cm[i,:].sum()-tp
    pr=tp/(tp+fp+1e-10); rc=tp/(tp+fn+1e-10)
    f1s.append(2*pr*rc/(pr+rc+1e-10))
ax2=axes[1]
bars=ax2.bar(class_names,f1s,color=['#B71C1C','#1565C0','#2E7D32'],edgecolor='white',lw=1.5)
ax2.set_ylim(0,1.1); ax2.set_ylabel('F1 Score')
ax2.set_title('Sınıf Bazlı F1',fontweight='bold')
for bar,v in zip(bars,f1s): ax2.text(bar.get_x()+bar.get_width()/2,v+.02,f'{v:.3f}',ha='center',fontweight='bold')
ax2.axhline(np.mean(f1s),color='k',ls='--',alpha=.7,label=f'Macro={np.mean(f1s):.3f}')
ax2.legend(); ax2.grid(True,alpha=.3,axis='y')
plt.tight_layout(); plt.show()
print(f"Macro F1: {np.mean(f1s):.3f}  |  Doğruluk: {(preds==y_te).mean():.2%}")


## 9. Sonuç

Bu final proje, Deep Learning Path boyunca öğrenilen tüm kavramları entegre etti:

| Modül | Kavram | Kullanım |
|-------|--------|----------|
| 01-04 | MLP, Backprop | Tüm Dense katmanları |
| 05 | Dropout, LayerNorm | Her dalda |
| 06-07 | CNN, Feature Extraction | Görüntü dalı |
| 08 | LSTM | Metin dalı |
| 09 | Attention (ort. pooling) | Metin aggregation |
| 10 | Latent fusion | Concat + projeksiyon |

**Deep Learning Path — 11 Modül tamamlandı!** 🎉
